# 13 GSE65858_clinical_variables_check

Source code preserved from the supplied analysis notebook. Saved outputs and execution counts have been cleared for public distribution. See README for input requirements and limitations.


In [ ]:
import pandas as pd
import numpy as np
import openpyxl

In [ ]:
wb = openpyxl.load_workbook("GSE65858.xlsx", data_only=True, read_only=True)
ws = wb["Sheet1"]

def get_row_values(row_idx, n_samples):
    row = list(ws.iter_rows(min_row=row_idx, max_row=row_idx, max_col=1+n_samples, values_only=True))[0]
    vals = []
    for v in row[1:]:
        if v is None:
            vals.append(None); continue
        v = str(v)
        vals.append(v.split(":", 1)[1].strip() if ":" in v else v)
    return vals

header_row = list(ws.iter_rows(min_row=105, max_row=105, values_only=True))[0]
sample_ids = [str(v) for v in header_row[1:] if v]
n = len(sample_ids)
print("Total samples in file:", n)

In [ ]:
fields = {
    "gender": 37, "age": 38, "smoking": 39, "packyears": 40, "alcohol": 41,
    "tumor_type": 42, "tumor_site": 43, "uicc_stage": 44, "t_category": 45,
    "n_category": 46, "distant_metastasis": 47, "treatment": 48,
    "hpv_dna": 49, "hpv16_dna_rna": 50,
    "os_days": 51, "os_event": 52,
}

data = {"Sample": sample_ids}
for name, row_idx in fields.items():
    data[name] = get_row_values(row_idx, n)

df = pd.DataFrame(data)
print("Raw extracted table:", df.shape)

In [ ]:
df["os_days_num"] = pd.to_numeric(df["os_days"], errors="coerce")
df["event_num"] = df["os_event"].map({"TRUE": 1, "FALSE": 0})
analysis_cohort = df[(df["os_days_num"] > 0) & (df["event_num"].notna())].copy()
print("Analysis cohort (valid survival data):", len(analysis_cohort))

In [ ]:
priority_vars = ["age", "uicc_stage", "tumor_site", "hpv_dna", "hpv16_dna_rna", "smoking"]
summary_rows = []
for v in priority_vars:
    missing = analysis_cohort[v].isna().sum() + (analysis_cohort[v] == "NA").sum()
    present = len(analysis_cohort) - missing
    pct = 100 * present / len(analysis_cohort)
    summary_rows.append({"Variable": v, "Present": present, "Missing": missing, "Pct_complete": round(pct, 1)})
    print(f"{v:20s}: {present}/{len(analysis_cohort)} present ({pct:.1f}%), {missing} missing")

missingness_summary = pd.DataFrame(summary_rows)
missingness_summary.to_csv("gse65858_missingness_summary.csv", index=False)

In [ ]:
for v in priority_vars:
    print(f"\n--- {v} ---")
    print(analysis_cohort[v].value_counts(dropna=False))

In [ ]:
analysis_cohort.to_csv("gse65858_clinical_variables.csv", index=False)
print("Saved gse65858_clinical_variables.csv:", analysis_cohort.shape)